In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from finrl.config import INDICATORS, TRAINED_MODEL_DIR, RESULTS_DIR
from stable_baselines3 import A2C, PPO, DDPG, TD3, SAC

# Pfade wie in 02_train (relativ zum Notebook-Ordner)
DATA_DIR = Path("../../data")
MODEL_DIR = Path(TRAINED_MODEL_DIR)                 # hier speichert 02_train die Agenten
OUT_DIR = Path(RESULTS_DIR) / "validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRADING_DAYS = 252          # Handelstage pro Jahr – Renditen sind pro Handelstag 
RISK_FREE_RATE = 0.0
SELECTION_METRIC = "Sharpe"
INDEX_TICKER = "^GDAXI"     # DAX-Performanceindex = passender Markt-Benchmark zum DAX-Universum
ALGO_CLASSES = {"a2c": A2C, "ppo": PPO, "ddpg": DDPG, "td3": TD3, "sac": SAC}

In [ ]:
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv

class TRStockTradingEnv(StockTradingEnv):
    """StockTradingEnv mit Trade-Republic-Fixgebuehr von 1 EUR je Ausfuehrung.

    Die geerbten buy_cost_pct / sell_cost_pct bleiben als Spread aktiv.
    Quelle der Kondition: Preis- und Leistungsverzeichnis Trade Republic.
    """

    def __init__(self, *args, fixed_fee=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.fixed_fee = fixed_fee
        self.fee_total = 0.0

    def _charge_fee(self, shares):
        if shares > 0:
            self.state[0] -= self.fixed_fee   # Cash belasten
            self.cost += self.fixed_fee       # in FinRLs Kostenzaehler
            self.fee_total += self.fixed_fee  # eigener Zaehler fuer die Auswertung

    def _buy_stock(self, index, action):
        shares = super()._buy_stock(index, action)
        self._charge_fee(shares)
        return shares

    def _sell_stock(self, index, action):
        shares = super()._sell_stock(index, action)
        self._charge_fee(shares)
        return shares

    def reset(self, *, seed=None, options=None):
        out = super().reset(seed=seed, options=options)
        self.fee_total = 0.0
        return out

In [ ]:
def load_like_train(file):
    df = pd.read_csv(DATA_DIR / file)
    date = df.columns[0]
    df.index = pd.factorize(df[date])[0]
    df.index.names = ['']
    return df

train = load_like_train("train.csv")
val = load_like_train("val.csv")

# Gleiches Ticker-Universum in gleicher Reihenfolge -> sonst passt der State nicht zum Agenten
assert list(train.tic.unique()) == list(val.tic.unique()), "Ticker in val.csv weichen von train.csv ab"

stock_dimensions = len(val.tic.unique())
state_space = 1 + 2 * stock_dimensions + len(INDICATORS) * stock_dimensions
print(f"Validierung: {val.date.min()} bis {val.date.max()} | {val.date.nunique()} Handelstage | "
      f"{stock_dimensions} Aktien | State Space {state_space}")

In [ ]:
buy_cost_list = sell_cost_list = [0.0005] * stock_dimensions
num_stock_shares = [0] * stock_dimensions

env_tweeks = {
    "hmax": 50,
    "initial_amount": 100000,
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimensions,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimensions,
    "reward_scaling": 1e-4
}
FIXED_FEE = 1.0
START = env_tweeks["initial_amount"]     # Startkapital auch für die Benchmarks
SPREAD = buy_cost_list[0]

In [ ]:
def kennzahlen(values: pd.Series) -> dict:
    r = values.pct_change().dropna()
    years = len(r) / TRADING_DAYS
    excess = r - RISK_FREE_RATE / TRADING_DAYS
    cagr = (values.iloc[-1] / values.iloc[0]) ** (1 / years) - 1
    vola = r.std() * np.sqrt(TRADING_DAYS)
    sharpe_d = excess.mean() / r.std() if r.std() > 0 else np.nan            # Tages-Sharpe
    downside = np.sqrt((np.minimum(excess, 0) ** 2).mean())                  # Downside Deviation
    mdd = (values / values.cummax() - 1).min()
    return {
        "Endwert": values.iloc[-1],
        "Rendite": values.iloc[-1] / values.iloc[0] - 1,
        "CAGR": cagr,
        "Vola": vola,
        "Sharpe": sharpe_d * np.sqrt(TRADING_DAYS),
        # Standardfehler der Sharpe Ratio (Lo 2002): zeigt, wie unsicher der Wert bei nur 1 Jahr Daten ist
        "Sharpe_SE": np.sqrt((1 + 0.5 * sharpe_d ** 2) / len(r)) * np.sqrt(TRADING_DAYS),
        "Sortino": excess.mean() / downside * np.sqrt(TRADING_DAYS) if downside > 0 else np.nan,
        "MaxDD": mdd,
        "Calmar": cagr / abs(mdd) if mdd < 0 else np.nan,
    }

In [ ]:
px = val.pivot(index="date", columns="tic", values="close").sort_index()
px.index = pd.to_datetime(px.index)
px = px.replace(0, np.nan).ffill().dropna(axis=1, how="any")

def equity(px, rebal_days, start=START, fee=FIXED_FEE, spread=SPREAD):
    """Depotwert je Handelstag. Rebalancing auf Gleichgewichtung an rebal_days."""
    n, rebal_days = px.shape[1], set(rebal_days)
    shares, cash, out = pd.Series(0.0, index=px.columns), start, []
    for d in px.index:
        p = px.loc[d]
        if d in rebal_days:
            value = cash + (shares * p).sum()
            trade_eur = ((value / n) / p - shares).abs() * p
            orders = int((trade_eur > 1.0).sum())                      # Mindestorder 1 EUR
            value -= orders * fee + trade_eur.sum() * spread
            shares, cash = (value / n) / p, 0.0
        out.append(cash + (shares * p).sum())
    return pd.Series(out, index=px.index)

def load_index(ticker, dates, start=START):
    """Index von Yahoo laden und auf Startkapital normieren. None, wenn der Download scheitert."""
    try:
        raw = yf.download(ticker, start=dates[0].strftime("%Y-%m-%d"),
                          end=(pd.Timestamp(dates[-1]) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
                          auto_adjust=True, progress=False)["Close"]
        s = raw.squeeze()
        s.index = pd.to_datetime(s.index)
        if s.index.tz is not None:
            s.index = s.index.tz_localize(None)
        s = s.reindex(dates).ffill().bfill()
        if s.isna().all():
            raise ValueError("keine Kurse")
        return s / s.iloc[0] * start
    except Exception as exc:
        print(f"{ticker} konnte nicht geladen werden ({exc}) -> ohne Index weiter")
        return None

monats_erste = px.groupby(px.index.to_period("M")).head(1).index

benchmarks = {
    "1/N halten":    equity(px, rebal_days=[px.index[0]]),   # einmal gleichgewichtet kaufen
    "1/N monatlich": equity(px, rebal_days=monats_erste),    # monatlich auf 1/N zurück
    "DAX":           load_index(INDEX_TICKER, px.index),
}
benchmarks = pd.DataFrame({k: v for k, v in benchmarks.items() if v is not None})
display(benchmarks.tail(3))

In [ ]:
def run_agent(model, data):
    """Agent läuft EINMAL deterministisch Tag für Tag durch data (nur predict, kein Lernen)."""
    env = TRStockTradingEnv(df=data, fixed_fee=FIXED_FEE, print_verbosity=10**9, **env_tweeks)
    obs, _ = env.reset()
    done = False
    while not done:
        action, _ = model.predict(np.asarray(obs, dtype=np.float32), deterministic=True)  # Agent entscheidet
        obs, _, terminated, truncated, _ = env.step(action)                                # Orders + nächster Tag
        done = terminated or truncated
    account = env.save_asset_memory()                     # Depotwert jedes Handelstages
    values = pd.Series(account["account_value"].to_numpy(), index=pd.to_datetime(account["date"]))
    # FinRLs env.trades zählt auch Kaufversuche ohne Ausführung -> echte Orders = Gebühren / 1 EUR
    stats = {"Orders": int(round(env.fee_total / FIXED_FEE)), "Gebuehren": env.fee_total, "Kosten": env.cost,
             "Cash_Ende": float(env.state[0])}
    return values, stats


def parse_name(path):
    """agent_ppo.zip -> ('ppo', 'ppo', None) | agent_ppo_s3.zip -> ('ppo_s3', 'ppo', 3)"""
    name = path.stem.removeprefix("agent_")
    algo, _, seed = name.partition("_s")
    return name, algo, int(seed) if seed.isdigit() else None


model_files = sorted(MODEL_DIR.glob("agent_*.zip"))
assert model_files, f"Keine Modelle in {MODEL_DIR.resolve()} - zuerst 02_train ausführen (inkl. Speichern)"

agent_values, agent_stats, agent_info = {}, {}, {}
for path in model_files:
    name, algo, seed = parse_name(path)
    if algo not in ALGO_CLASSES:
        print(f"Übersprungen (unbekannter Algo): {path.name}")
        continue
    model = ALGO_CLASSES[algo].load(str(path), device="cpu")
    if model.observation_space.shape[0] != state_space:
        print(f"Übersprungen: {path.name} erwartet State {model.observation_space.shape[0]}, Daten liefern {state_space}")
        continue
    agent_values[name], agent_stats[name] = run_agent(model, val)
    agent_info[name] = {"Algo": algo, "Seed": seed, "Datei": path.name}
    print(f"{name:<8} Endwert {agent_values[name].iloc[-1]:>12,.0f} | Orders {agent_stats[name]['Orders']:>5}")

values = pd.concat([benchmarks, pd.DataFrame(agent_values)], axis=1).sort_index().ffill()


In [ ]:
metrics = pd.DataFrame({col: kennzahlen(values[col].dropna()) for col in values}).T
metrics = metrics.join(pd.DataFrame(agent_stats).T).join(pd.DataFrame(agent_info).T)
metrics["Typ"] = np.where(metrics.index.isin(benchmarks.columns), "Benchmark", "Agent")

pd.options.display.float_format = "{:,.3f}".format
display(metrics.sort_values(SELECTION_METRIC, ascending=False))

In [ ]:
#   mehrere Seeds je Algo: 1) Algo mit bestem MEDIAN  2) bester Seed in diesem Algo
#   ein Modell je Algo:    einfach das Modell mit der besten Kennzahl
agents = metrics[metrics["Typ"] == "Agent"].copy()
agents[SELECTION_METRIC] = agents[SELECTION_METRIC].astype(float).fillna(0.0)   # nie gehandelt -> 0

algo_rank = (agents.groupby("Algo")[SELECTION_METRIC]
             .agg(["median", "min", "max", "count"])
             .sort_values("median", ascending=False))
best_algo = algo_rank.index[0]
best = (agents[agents["Algo"] == best_algo]
        .sort_values([SELECTION_METRIC, "Calmar"], ascending=False).index[0])

best_bench_name = metrics.loc[benchmarks.columns, SELECTION_METRIC].astype(float).idxmax()
best_bench = metrics.loc[best_bench_name, SELECTION_METRIC]
best_val, best_se = metrics.loc[best, SELECTION_METRIC], metrics.loc[best, "Sharpe_SE"]

display(algo_rank)
print(f"Gewählter Agent: {best} | Sharpe {best_val:.2f} ± {best_se:.2f} (1 SE) | "
      f"bester Benchmark {best_bench_name}: {best_bench:.2f}")
if best_val <= best_bench:
    print("-> Kein Agent schlägt den besten Benchmark auf dem Validierungsdatensatz.")
elif best_val - best_bench < best_se:
    print("-> Vorsprung kleiner als 1 Standardfehler: statistisch nicht von Benchmark unterscheidbar.")

In [ ]:
ALGO_COLORS = {"ppo": "#D55E00", "a2c": "#CC79A7", "ddpg": "#56B4E9", "td3": "#B8A000", "sac": "#000000"}
BENCH_COLORS = {"1/N halten": "#0072B2", "1/N monatlich": "#E69F00", "DAX": "#009E73"}

def color_of(name):
    return BENCH_COLORS.get(name) or ALGO_COLORS.get(name.partition("_s")[0], "#666666")

def plot_lines(df, title, ylabel, filename):
    fig, ax = plt.subplots(figsize=(15, 7))
    for col in df.columns:
        is_bench, is_best = col in BENCH_COLORS, col == best
        ax.plot(df[col], color=color_of(col), linestyle="--" if is_bench else "-",
                linewidth=3 if (is_bench or is_best) else 1.2,
                alpha=1 if (is_bench or is_best) else 0.6,
                label=f"{col}  << gewählt" if is_best else col)
    ax.set(title=title, xlabel="Datum", ylabel=ylabel)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
    fig.tight_layout()
    fig.savefig(OUT_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()

plot_lines(values / values.iloc[0] * 100, "Validierung: Portfolioentwicklung (Start = 100)",
           "Indexierter Portfoliowert", "equity_val.png")
plot_lines(values / values.cummax() - 1, "Validierung: Drawdown", "Drawdown", "drawdown_val.png")

In [ ]:
order = metrics.sort_values(SELECTION_METRIC).index
fig, ax = plt.subplots(figsize=(10, 0.4 * len(order) + 2))
ax.barh(order, metrics.loc[order, SELECTION_METRIC].astype(float),
        xerr=metrics.loc[order, "Sharpe_SE"].astype(float), capsize=3,
        color=[color_of(n) for n in order])
ax.axvline(best_bench, color="grey", ls="--", label=f"bester Benchmark ({best_bench:.2f})")
ax.set(title="Validierung: Sharpe Ratio ± 1 Standardfehler", xlabel="Sharpe (annualisiert)")
ax.grid(axis="x", alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "sharpe_val.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 6))
for name in metrics.index:
    ax.scatter(metrics.loc[name, "MaxDD"], metrics.loc[name, "Rendite"], color=color_of(name), s=90,
               marker="s" if name in BENCH_COLORS else "o", edgecolors="black" if name == best else "none")
    ax.annotate(name, (metrics.loc[name, "MaxDD"], metrics.loc[name, "Rendite"]),
                xytext=(5, 4), textcoords="offset points", fontsize=8)
ax.set(title="Validierung: Rendite vs. Max Drawdown", xlabel="Max Drawdown", ylabel="Gesamtrendite")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "return_vs_dd_val.png", dpi=150, bbox_inches="tight")
plt.show()

# Seed-Streuung – nur sinnvoll, wenn es mehrere Seeds je Algo gibt
if (algo_rank["count"] > 1).any():
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, (algo, group) in enumerate(agents.groupby("Algo")):
        ax.scatter([i] * len(group), group[SELECTION_METRIC], color=ALGO_COLORS.get(algo, "#666666"), s=70, zorder=3)
        ax.hlines(group[SELECTION_METRIC].median(), i - 0.3, i + 0.3, color="black", linewidth=2)
    ax.set_xticks(range(agents["Algo"].nunique()), sorted(agents["Algo"].unique()))
    ax.axhline(best_bench, color="grey", ls="--", label="bester Benchmark")
    ax.set(title="Validierung: Sharpe je Seed (Strich = Median)", ylabel="Sharpe")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT_DIR / "seed_spread_val.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
selection = {
    "name": best,
    "algo": best_algo,
    "seed": None if pd.isna(agents.loc[best, "Seed"]) else int(agents.loc[best, "Seed"]),
    "file": agents.loc[best, "Datei"],
    "model_path": str((MODEL_DIR / agents.loc[best, "Datei"]).resolve()),
    "selected_by": SELECTION_METRIC,
    "val_period": [str(val.date.min()), str(val.date.max())],
    "val_metrics": {k: float(metrics.loc[best, k]) for k in
                    ["Endwert", "Rendite", "CAGR", "Vola", "Sharpe", "Sharpe_SE", "Sortino", "MaxDD",
                     "Calmar", "Orders", "Gebuehren"]},
    "best_benchmark": {"name": best_bench_name, "Sharpe": float(best_bench)},
}
(OUT_DIR / "best_agent.json").write_text(json.dumps(selection, indent=2, ensure_ascii=False), encoding="utf-8")
values.to_csv(OUT_DIR / "portfolio_values_val.csv")
metrics.to_csv(OUT_DIR / "metrics_val.csv")
algo_rank.to_csv(OUT_DIR / "algo_ranking_val.csv")

print(f"Gespeichert in {OUT_DIR.resolve()}")
print(json.dumps(selection, indent=2, ensure_ascii=False))